In [39]:
import json
import os
import math
from uuid import uuid4

# ============================================================
# 1) PERCORSI FILE ORIGINALI (NON PULITI)
#    (USA albums.json e artists_filled_with_feat.json)
# ============================================================

artists_in = r"C:\Users\Win10\OneDrive - Università degli Studi di Torino\Desktop\repo_dss\dss_lab_project\dataset\filled\artists_filled_with_feat.json"
tracks_in  = r"C:\Users\Win10\OneDrive - Università degli Studi di Torino\Desktop\repo_dss\dss_lab_project\dataset\filled\albums.json"

# Cartella di output per i file finali
output_dir = r"C:\Users\Win10\OneDrive - Università degli Studi di Torino\Desktop\repo_dss\dss_lab_project\dataset\Finali"
os.makedirs(output_dir, exist_ok=True)

artists_out = os.path.join(output_dir, "artists_final.json")
tracks_out  = os.path.join(output_dir, "tracks_final.json")

# ============================================================
# 2) CARICAMENTO FILE ORIGINALI
# ============================================================

with open(artists_in, "r", encoding="utf-8") as f:
    artists = json.load(f)

with open(tracks_in, "r", encoding="utf-8") as f:
    tracks = json.load(f)

print(f"Caricati: {len(artists)} artists, {len(tracks)} tracks")


# ============================================================
# 3) FUNZIONE PER RICONOSCERE VALORI MISSING/SPORCHI
# ============================================================

def is_raw_missing(v):
    """Valori da considerare missing PRIMA della pulizia."""
    if v is None:
        return True
    if isinstance(v, float) and math.isnan(v):
        return True
    if isinstance(v, str) and v.strip().lower() in ["", "none", "null", "nan", "undefined"]:
        return True
    return False


def normalize_missing(v):
    """Converte tutti i missing in stringa 'NULL'."""
    if is_raw_missing(v):
        return "NULL"
    return v


# ============================================================
# 4) PULIZIA COMPLETA DI ARTISTS E TRACKS
# ============================================================

def clean_dataset(data):
    for row in data:
        for col, val in row.items():
            row[col] = normalize_missing(val)

clean_dataset(artists)
clean_dataset(tracks)

print("✔ Pulizia generale completata (tutti i missing → 'NULL').")


# ============================================================
# 5) FIX SPECIFICO COLONNA 'type' IN ARTISTS
#    - Se manca la colonna: la creo con 'NULL'
#    - Se è sporca (raw): la porto a 'NULL'
# ============================================================

for a in artists:
    if "type" not in a:
        a["type"] = "NULL"
    else:
        a["type"] = normalize_missing(a["type"])

print("✔ Colonna 'type' sistemata in tutti gli artists.")


# ============================================================
# 6) GENERAZIONE ID LYRICS / SYMPHONY / GEO (se vuoi i nuovi ID)
# ============================================================

# new_id_lyrics e new_id_symphony in tracks
for t in tracks:
    t["new_id_lyrics"] = str(uuid4())
    t["new_id_symphony"] = str(uuid4())

# new_id_geo in artists
for a in artists:
    a["new_id_geo"] = str(uuid4())

print("✔ Generati new_id_lyrics, new_id_symphony nelle tracks e new_id_geo negli artists.")


# ============================================================
# 7) SALVATAGGIO FILE FINALI PULITI
# ============================================================

with open(artists_out, "w", encoding="utf-8") as f:
    json.dump(artists, f, indent=4, ensure_ascii=False)

with open(tracks_out, "w", encoding="utf-8") as f:
    json.dump(tracks, f, indent=4, ensure_ascii=False)

print("\n✔ File PULITI salvati in:")
print(" -", artists_out)
print(" -", tracks_out)


# ============================================================
# 8) CHECK FINALE SU 'type' (DOPO LA PULIZIA)
#    ATTENZIONE: QUI 'NULL' È PULITO, NON È ERRORE
# ============================================================

def is_dirty_after_clean(v):
    """Dopo la pulizia: un valore è sporco SOLO se è ancora None, NaN o stringa vuota, etc.
       'NULL' è considerato OK."""
    if v is None:
        return True
    if isinstance(v, float) and math.isnan(v):
        return True
    if isinstance(v, str) and v.strip().lower() in ["", "none", "nan", "undefined"]:
        return True
    # 'NULL' è valido → False
    return False


dirty_type = [a["type"] for a in artists if is_dirty_after_clean(a.get("type"))]

print("\n=== CHECK FINALE SU 'type' ===")
print("Valori sporchi residui in 'type':", len(dirty_type))
if dirty_type:
    print("Esempi:", dirty_type[:10])
else:
    print("✔ Nessun valore sporco residuo in 'type'. Tutto OK.")


Caricati: 104 artists, 11166 tracks
✔ Pulizia generale completata (tutti i missing → 'NULL').
✔ Colonna 'type' sistemata in tutti gli artists.
✔ Generati new_id_lyrics, new_id_symphony nelle tracks e new_id_geo negli artists.

✔ File PULITI salvati in:
 - C:\Users\Win10\OneDrive - Università degli Studi di Torino\Desktop\repo_dss\dss_lab_project\dataset\Finali\artists_final.json
 - C:\Users\Win10\OneDrive - Università degli Studi di Torino\Desktop\repo_dss\dss_lab_project\dataset\Finali\tracks_final.json

=== CHECK FINALE SU 'type' ===
Valori sporchi residui in 'type': 0
✔ Nessun valore sporco residuo in 'type'. Tutto OK.


In [37]:
""" # salvataggio file puliti
import os
import json
# =======================
# 3) SALVATAGGIO NEI FILE FINALI
# =======================

output_dir = r"C:\Users\Win10\OneDrive - Università degli Studi di Torino\Desktop\repo_dss\dss_lab_project\dataset\Finali"

# crea la cartella se non esiste
os.makedirs(output_dir, exist_ok=True)

artists_out = os.path.join(output_dir, "artistsNew.json")
tracks_out  = os.path.join(output_dir, "tracksNew.json")

with open(artists_out, "w", encoding="utf-8") as f:
    json.dump(artists, f, indent=4, ensure_ascii=False)

with open(tracks_out, "w", encoding="utf-8") as f:
    json.dump(tracks, f, indent=4, ensure_ascii=False)

print(f"✔ File salvati correttamente in:\n{artists_out}\n{tracks_out}") 
""" 

SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 152-153: truncated \UXXXXXXXX escape (1403326668.py, line 1)